# Task 1b — does substitution *find* constituents?

**Run all cells.** Everything below is idempotent and resumable: re-running a cell is always safe,
and the expensive scoring phase picks up where it left off after a runtime disconnect.

The question. Acts 3–4 of the teaching doc showed that when *we* name a span, the model's fluency
drop tells real constituents from fake ones. Here nobody names anything: every contiguous span of
every sentence is replaced by a proform, the fluency drop is its *cost*, a tree is induced from the
cheapest non-crossing spans, and that tree is scored against the treebank's gold brackets. If the
theory in `theory_1b_bracket_induction.md` is unfamiliar, read it first.

What this notebook does:

1. Clones the private repo `SalmonSung/m1_llms_analyzer` and installs its dependencies.
2. Runs a **smoke test** of the whole experiment on a ~5 MB model and three hand-parsed sentences.
3. Loads **N treebank sentences** (NLTK's free Penn Treebank sample) with their gold brackets.
4. Loads the model with its LM head and walks through **one sentence** end to end.
5. **Phase A (GPU):** scores every span × proform variant, caching raw numbers to a resumable JSONL file.
6. **Phase B (CPU):** induces a bracketing per sentence, computes unlabelled F1 against gold and
   against the right-branching, left-branching and random baselines with bootstrap intervals,
   F1 by sentence length, and the pooled rank curve.
7. Writes the `fig_1b` record, draws `fig_1b` and `fig_0c`, and persists everything to Drive.

**Pass:** substitution-induced F1 clears both trivial baselines by its interval, at every length,
and the rank curve bows well above the diagonal.
**Fail:** it ties or loses to right-branching — the test confirms boundaries, it does not find them.

---

## Before you run: two Colab secrets

Open the **key icon** in the left sidebar and add:

| Secret | Required? | What it is |
|---|---|---|
| `GITHUB_TOKEN` | **Yes** — the repo is private | A GitHub fine-grained PAT with *Contents: Read* on this repo. [Create one](https://github.com/settings/personal-access-tokens/new) |
| `HF_TOKEN` | Only for gated models | A Hugging Face read token. [Create one](https://huggingface.co/settings/tokens) |

Toggle **Notebook access** on for each secret. The default model (`Qwen/Qwen3-0.6B-Base`) is
ungated. Use a **GPU runtime** (Runtime → Change runtime type → T4): phase A is ~700k short
sequences for 1000 sentences, about half an hour on a T4 and hours on a CPU.

Neither token is ever printed, logged, or written into any output file.

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

**This is the only cell you normally edit.**

| Setting | Meaning |
|---|---|
| `MODEL_ID` | Any decoder-only Hugging Face model id. Use a **base** model, not an instruct one: the cost is a raw-text log-probability, and chat tuning distorts it. |
| `DTYPE` | `auto` picks fp16 on a T4. Switch to `float32` if phase A reports non-finite failures. |
| `N_SENTENCES`, `MIN_LEN`, `MAX_LEN` | How many treebank sentences, and the word-count window (after punctuation removal). |
| `PROFORMS` | The blind replacement set; every span is scored with each and the cheapest wins. All are cached, so a fixed-by-length policy can be re-read later without the GPU (`length:2-3=it,4-6=that,7+=this`). Each proform set gets its own cache, record, and figure files, so several sets can be run on one model without overwriting each other. Multi-word entries (`do so`) are fine. |
| `CONTROLS` | Replacements scored and cached for every span **but never chosen** for tree induction: `blorp` (a category-less nonsense word, the floor every real proform should beat) and `<del>` (delete the span outright, so "any shortening helps the mean" can be separated from "the proform fits"). |
| `INCLUDE_TREES` | Store each sentence's original bracketed parse in the cache row, so gold can be re-derived under other conventions later without the corpus. |
| `INDUCER` | `greedy` (cheapest first, never cross — the theory doc) or `cky` (minimum total cost). |
| `BATCH_SIZE` | `"auto"` probes the GPU once (the largest batch of the longest text that fits, so a T4 and an L4 each get their own size), then adapts: halves on out-of-memory and remembers it, grows back after a run of clean batches. An integer pins the size (it still halves on OOM, but never grows). |
| `RESUME` | Keep the existing cost cache and score only what is missing. Set `False` to rescore from scratch. |
| `MIRROR_TO_DRIVE` | Copy the cache to Drive every few sentences, so a recycled runtime loses nothing. |

In [ ]:
#@title Run configuration { display-mode: "form" }
SMOKE_MODEL_ID  = "sshleifer/tiny-gpt2"        #@param {type:"string"}
MODEL_ID        = "Qwen/Qwen3-0.6B-Base"       #@param {type:"string"}
DTYPE           = "auto"                       #@param ["auto", "float16", "bfloat16", "float32"]
N_SENTENCES     = 1000                         #@param {type:"integer"}
MIN_LEN         = 5                            #@param {type:"integer"}
MAX_LEN         = 30                           #@param {type:"integer"}
PROFORMS        = "it,there,did,then,do so,does so,did so,done so,doing so,is,was,be,been,happens,happened"  #@param {type:"string"}
CONTROLS        = "blorp,<del>"                #@param {type:"string"}
INCLUDE_TREES   = True                         #@param {type:"boolean"}
INDUCER         = "greedy"                     #@param ["greedy", "cky"]
N_RANDOM        = 10                           #@param {type:"integer"}
BATCH_SIZE      = "auto"                       #@param {type:"raw"}
OUTPUT_DIR      = "outputs/task_1b"            #@param {type:"string"}
SEED            = 42                           #@param {type:"integer"}
RESUME          = True                         #@param {type:"boolean"}
MIRROR_TO_DRIVE = False                        #@param {type:"boolean"}
DRIVE_DIR       = "/content/drive/MyDrive/m1_llms_analyzer/task_1b"  #@param {type:"string"}

import re
from pathlib import Path

from m1_analyzer import ModelConfig, RunConfig, ScoringConfig, StorageConfig
from m1_analyzer.experiments import parse_policy

POLICY = parse_policy(PROFORMS, controls=CONTROLS)
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)


def slug(text: str) -> str:
    """Filesystem-safe name (model id, policy name) for output files."""
    return re.sub(r"-{2,}", "-", re.sub(r"[^A-Za-z0-9._-]+", "-", text)).strip("-")


def make_config(model_id: str, **overrides) -> RunConfig:
    """A RunConfig with the LM head, so `Analyzer.score()` works."""
    return RunConfig(
        model=ModelConfig(model_id=model_id, head="causal_lm", dtype=overrides.get("dtype", DTYPE)),
        scoring=ScoringConfig(batch_size=overrides.get("batch_size", BATCH_SIZE), max_batch_size=512),
        storage=StorageConfig(output_dir=str(OUT)),
        seed=SEED,
    )


# Every output is keyed by model AND policy, so a different proform set on the same model
# gets its own cache, record, and figures (locally and on Drive) instead of a header clash.
RUN_TAG = f"{slug(MODEL_ID)}_{slug(POLICY.name)}"
CACHE_PATH = OUT / f"span_costs_{RUN_TAG}.jsonl"
RECORD_PATH = OUT / f"record_1b_{RUN_TAG}.json"
FIG_1B_PATH = OUT / f"fig_1b_{RUN_TAG}.png"
FIG_0C_PATH = OUT / f"fig_0c_{RUN_TAG}.png"

print(f"policy   : {POLICY.name}")
print(f"variants : {len(POLICY.proforms)} per span ({len(POLICY.controls)} of them controls)")
print(f"cache    : {CACHE_PATH}")
print("Configuration ready.")

## 5 · Smoke test (~5 MB, a few seconds)

Runs the *entire* experiment — scoring, caching, induction, baselines, bootstrap, figure — on a
tiny model and three hand-parsed sentences. If this passes, the only things that can still go
wrong with the real model are download size, GPU memory, and the treebank download.

In [ ]:
#@title Smoke test on a tiny model
import time

import matplotlib
matplotlib.use("Agg")

from m1_analyzer import Analyzer
from m1_analyzer.experiments import experiment_figures as EF
from m1_analyzer.experiments import hand_examples, run_task_1b, validate_record, verdict

smoke = Analyzer(make_config(SMOKE_MODEL_ID, batch_size=16))
SMOKE_CACHE = OUT / f"smoke_span_costs_{slug(POLICY.name)}.jsonl"
print(smoke.describe()["head"], "|", smoke.models.architecture, "| BOS token:", smoke.scoring.bos_token())

started = time.perf_counter()
smoke_record, smoke_tables = run_task_1b(
    smoke, hand_examples(), policy=POLICY, cache_path=SMOKE_CACHE,
    provenance={"model_id": SMOKE_MODEL_ID}, model=SMOKE_MODEL_ID, min_per_length=1, show_progress=False,
)
validate_record(smoke_record)
print(f"\n{len(smoke_tables)} sentences scored and analysed in {time.perf_counter() - started:.1f}s")
for m in smoke_record["methods"]:
    print(f"  {m['name']:<24} F1 {m['f1']:.3f}  CI {m['ci']}")
EF.fig_1b(smoke_record, path=str(OUT / "smoke_fig_1b.png"))
assert (OUT / "smoke_fig_1b.png").stat().st_size > 10_000
assert SMOKE_CACHE.exists()
print("\nSmoke test passed (the numbers above are meaningless: a random 5 MB model).")

smoke.unload()   # free the tiny model before loading the real one

## 6 · The treebank

NLTK ships 10% of the Penn Treebank's *Wall Street Journal* section (3,914 sentences) with the
annotators' constituency trees — real gold, freely downloadable. Each tree is flattened to the set
of spans a scorer sees: traces (`-NONE-`) and punctuation removed, unaries collapsed, labels
dropped, single words and the whole sentence excluded. The record's `treebank` field names this.

In [ ]:
#@title Load N sentences with their gold brackets
import collections
import time

from m1_analyzer.experiments import PTB_NLTK_NAME, load_ptb_nltk, spans_to_brackets, variant_count

started = time.perf_counter()
sentences = load_ptb_nltk(N_SENTENCES, min_len=MIN_LEN, max_len=MAX_LEN, seed=SEED)
print(f"{len(sentences)} sentences from {PTB_NLTK_NAME} in {time.perf_counter() - started:.1f}s")

lengths = collections.Counter(s.n for s in sentences)
print("words :", " ".join(f"{k}:{v}" for k, v in sorted(lengths.items())))
print(f"mean  : {sum(s.n for s in sentences) / len(sentences):.1f} words, "
      f"{sum(len(s.gold_spans) for s in sentences) / len(sentences):.1f} gold spans per sentence")

example = sentences[0]
print(f"\n{example.id}: {example.text}")
print("gold  :", spans_to_brackets(example.words, example.gold_spans))

n_variants = variant_count(sentences, POLICY)
print(f"\nPhase A will score {n_variants:,} texts "
      f"(~{n_variants / 400 / 60:.1f} min on a T4 for a 0.6B model in fp16; ~{n_variants / 1500 / 60:.1f} min for GPT-2 124M).")

## 7 · Load the model

First download takes a minute; it is cached for the rest of the session. The model is loaded
**with its language-model head** (`head="causal_lm"`), which is what turns text into
log-probabilities. If it is gated (Llama, Gemma, ...), accept its licence on the Hub and add
`HF_TOKEN` to Colab secrets.

In [ ]:
#@title Load the model
import time

from m1_analyzer import Analyzer

started = time.perf_counter()
analyzer = Analyzer(make_config(MODEL_ID))
print(f"Loaded in {time.perf_counter() - started:.1f}s\n")
for key, value in analyzer.describe().items():
    if key != "layers":
        print(f"{key:>18} : {value}")
print(f"{'BOS token':>18} : {analyzer.scoring.bos_token()!r} (prepended so every real token is predicted)")

PROVENANCE = {
    "model_id": MODEL_ID,
    "revision": analyzer.models.metadata()["revision"],
    "dtype": analyzer.models.metadata()["dtype"],
    "bos_token": analyzer.scoring.bos_token(),
    "treebank": PTB_NLTK_NAME,
    "min_len": MIN_LEN, "max_len": MAX_LEN, "seed": SEED, "n_sentences": len(sentences),
}

## 8 · One sentence, end to end

The theory doc's own sentence, with its seven hand-written gold spans. This is §4–5 of that doc
run live: every span scored, the cheapest-first walk, the induced tree against gold. On GPT-2 124M
the doc reports base fluency 3.8897 nats/token and F1 0.56 vs 0.33 for right-branching; on any
other model the numbers differ, but the procedure is the same.

In [ ]:
#@title Walk through the theory doc's sentence
from m1_analyzer.experiments import (
    bracket_prf, compute_span_costs, greedy_induce, right_branching, spans_to_brackets, theory_example,
)

one = theory_example()
print(one.text)
base = analyzer.score_one(one.text)
print(f"base fluency : {-base.mean_logprob:.4f} nats/token over {base.n_tokens} predicted tokens\n")

[table] = compute_span_costs(analyzer, [one], POLICY, show_progress=False)
costs = table.costs_for(POLICY)
kept, trace = greedy_induce(costs, one.n, trace=True)

print("cheapest five spans (cost in nats/token, per proform):")
for span, cost, blocker in trace[:5]:
    per = ", ".join(f"{p} {c:.2f}" for p, c in sorted(table.costs_by_proform(span).items(), key=lambda kv: kv[1]))
    tag = "kept" if blocker is None else f"rejected: crosses {' '.join(one.words[blocker[0]:blocker[1] + 1])!r}"
    print(f"  {cost:5.2f}  {' '.join(one.words[span[0]:span[1] + 1])!r:<45} [{per}]  {tag}")

p, r, f1, match = bracket_prf(kept, one.gold_spans)
print(f"\ninduced : {spans_to_brackets(one.words, kept)}")
print(f"gold    : {spans_to_brackets(one.words, one.gold_spans)}")
print(f"\n{len(kept)} induced spans, {len(one.gold_spans)} gold, {match} in both -> "
      f"P {p:.2f} R {r:.2f} F1 {f1:.2f}   (right-branching {bracket_prf(right_branching(one.n), one.gold_spans)[2]:.2f})")

## 9 · Phase A — score every span (GPU)

Every span × replacement variant of every sentence (the proforms *and* the controls), in
length-sorted batches. The raw log-probability sum and token count of each variant go to a JSONL
cache, one line per finished sentence, so a runtime disconnect costs at most one sentence: re-run
this cell and it resumes. Each line also carries the sentence's answer key (gold spans and, with
`INCLUDE_TREES`, its bracketed parse), so the cache alone is a complete record of the run. It
refuses to mix models or replacement sets (it names the mismatching field).

With `BATCH_SIZE = "auto"` the first thing this cell does is probe the GPU with the longest
variant of the run, doubling the batch until it no longer fits; the size it settles on is
printed next to the GPU name. A T4 and an L4 therefore get different sizes with no edit.


In [ ]:
#@title Score all spans (resumable)
import shutil
import time

import torch

from m1_analyzer.experiments import compute_span_costs, detokenize_ptb, ptb_tree_strings, substitute

if not RESUME and CACHE_PATH.exists():
    CACHE_PATH.unlink()
    print("RESUME=False: deleted the existing cache; scoring from scratch.")

mirror = None
if MIRROR_TO_DRIVE:
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
    mirror = Path(DRIVE_DIR) / CACHE_PATH.name
    if not CACHE_PATH.exists() and mirror.exists():
        shutil.copy(mirror, CACHE_PATH)
        print(f"Restored cache from Drive: {mirror}")

trees = ptb_tree_strings(sentences) if INCLUDE_TREES else None

if BATCH_SIZE == "auto":
    # Probe with the worst case: the longest sentences carrying the longest replacement.
    longest = sorted(sentences, key=lambda s: len(s.text))[-10:]
    widest = max(POLICY.proforms, key=len)
    probe_texts = [detokenize_ptb(substitute(s.words, 1, 1, widest)) for s in longest]
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    chosen = analyzer.scoring.calibrate_batch_size(probe_texts)
    print(f"batch size : {chosen} on {gpu} (calibrated; halves on OOM and remembers it, "
          f"grows back after clean batches up to {analyzer.scoring.batch_sizer.ceiling})")

started = time.perf_counter()
tables = compute_span_costs(
    analyzer, sentences, POLICY, cache_path=CACHE_PATH, provenance=PROVENANCE,
    show_progress=True, mirror_path=mirror, batch_size=BATCH_SIZE, trees=trees,
)
elapsed = time.perf_counter() - started
failed = sum(t.failed for t in tables)
print(f"\n{len(tables)} / {len(sentences)} sentences scored in {elapsed / 60:.1f} min; "
      f"{failed} variant(s) failed; cache {CACHE_PATH.stat().st_size / 1024**2:.1f} MB")
if analyzer.scoring.batch_sizer is not None:
    sizer = analyzer.scoring.batch_sizer
    print(f"batch size ended at {sizer.current} ({sizer.shrinks} halving(s), {sizer.grows} doubling(s))")
if failed:
    print("Non-finite scores usually mean a float16 overflow: set DTYPE='float32' and re-run with RESUME=False.")

## 10 · Phase B — induce, score, compare (CPU)

Reads the cache, reduces each span's proform costs to one cost under the policy, induces a
bracketing, and scores it. `methods` is the sentence-level mean F1 with a 95% bootstrap interval
over sentences; the corpus-level micro F1 the literature also reports is in `diagnostics`. This
cell is seconds, so try `INDUCER="cky"` or another policy for free.

In [ ]:
#@title Analyse
from m1_analyzer.experiments import analyse_1b, verdict

record = analyse_1b(
    tables, sentences, policy=POLICY, inducer=INDUCER, n_random=N_RANDOM, seed=SEED,
    model=MODEL_ID, treebank=PTB_NLTK_NAME,
    notes=f"dtype={PROVENANCE['dtype']}, bos={PROVENANCE['bos_token']!r}",
)
print(f"{record['n_sentences']} sentences, {record['treebank']}\n")
for m in record["methods"]:
    print(f"  {m['name']:<24} F1 {m['f1']:.3f}   95% CI [{m['ci'][0]:.3f}, {m['ci'][1]:.3f}]")
d = record["diagnostics"]
print(f"\n  corpus micro F1 {d['micro']['f1']:.3f} | gap vs right-branching {d['gap_vs_rb']['diff']:+.3f} "
      f"{d['gap_vs_rb']['ci']} | rank-curve area {d['rank_area']:.3f}\n")
print(verdict(record))

## 11 · Figures

In [ ]:
#@title Draw fig_1b and fig_0c
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

from m1_analyzer.experiments import experiment_figures as EF

plt.close(EF.fig_1b(record, path=str(FIG_1B_PATH)))
plt.close(EF.fig_0c(record, path=str(FIG_0C_PATH)))
display(Image(filename=str(FIG_1B_PATH)))
display(Image(filename=str(FIG_0C_PATH)))
print(FIG_1B_PATH, "\n", FIG_0C_PATH)

## 12 · Save and persist

The record is the experiment's single output: everything `fig_1b` needs plus per-sentence
diagnostics. It is written atomically. Colab wipes `/content` when the runtime is recycled, so the
Drive cell copies the record, the figures, and the cost cache (the expensive part) to `MyDrive`.

The cache is self-contained: every line carries the sentence's words, gold spans, provenance
and (with `INCLUDE_TREES`) its bracketed parse next to the raw scores, so
`load_span_costs_with_gold(path)` rebuilds tables *and* sentences on any machine, no treebank
installed. There is no separate gold file to keep paired with it.


In [ ]:
#@title Save the record
import json
import os
import tempfile

def atomic_write_json(path, payload):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=f".{path.stem}.", suffix=".json")
    os.close(fd)
    Path(tmp).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

atomic_write_json(RECORD_PATH, record)
print(f"record : {RECORD_PATH} ({RECORD_PATH.stat().st_size / 1024:.1f} KB)")
print(json.dumps({k: record[k] for k in ("n_sentences", "treebank", "methods", "meta")}, indent=2))

In [ ]:
#@title Persist results to Google Drive (survives a runtime disconnect)
import shutil

from m1_analyzer import in_colab

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DIR)
    target.mkdir(parents=True, exist_ok=True)
    for path in (RECORD_PATH, FIG_1B_PATH, FIG_0C_PATH, CACHE_PATH):
        if path.exists():
            shutil.copy(path, target / path.name)
            print("copied", target / path.name)
else:
    print("Not running in Colab -- skipping Drive mount.")

In [ ]:
#@title Download the record and figures to your machine
from m1_analyzer import in_colab

if in_colab():
    from google.colab import files
    for path in (RECORD_PATH, FIG_1B_PATH, FIG_0C_PATH):
        files.download(str(path))
else:
    print("Not in Colab; the files are already on disk under", OUT)

In [ ]:
#@title Run the test suite inside Colab
import subprocess, sys

result_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True
)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `GITHUB_TOKEN is not set` / `401` / `403` / `cannot see <repo>` | See the bootstrap cell's message: add or fix the fine-grained PAT (Contents: Read on this repo), enable *Notebook access*, re-run cell 1. |
| `Resource 'treebank' not found` | The NLTK download failed (no network, or a proxy). Re-run the treebank cell; on a proxied machine set `NLTK_ALLOW_PROXIED_URLOPEN=1`. |
| `Access to '<model>' was denied` | The model is gated: accept its licence on its Hub page, then add `HF_TOKEN` to Colab secrets. |
| `has no causal language-model head` | The model id is an encoder (BERT-like) or encoder-decoder. Use a decoder-only base model. |
| `N variant(s) failed` with *non-finite* errors | float16 overflowed inside the model. Set `DTYPE = "float32"`, `RESUME = False`, and re-run phase A. |
| `CUDA out of memory` | Batches halve automatically and the halved size sticks for the rest of the run, so this only appears if a *single* text does not fit. `ScoringConfig(logit_chunk=2)` shrinks the float32 logit block; a smaller model is the other fix. |
| Phase A is slow and the log shows `Batch size N -> N/2` repeatedly | The GPU changed (an L4 size on a T4) or the model is bigger than last time. Leave `BATCH_SIZE = "auto"`: the probe picks the size for *this* GPU; a pinned integer never grows. |
| `was written for model_id=... but this run has ...` | The cache belongs to another run. File names include the model and policy, so this only happens when an old cache was renamed or copied by hand. Change `OUTPUT_DIR`, or set `RESUME = False` to delete it. |
| Phase A restarts from zero after a disconnect | `/content` was wiped. Set `MIRROR_TO_DRIVE = True` so the cache is copied to Drive as it grows and restored on the next run. |
| Panel B is empty | Fewer than 5 sentences per length. Raise `N_SENTENCES` or narrow `MIN_LEN..MAX_LEN`. |
| Re-analysing elsewhere without NLTK | Take the one `span_costs_*.jsonl`; `load_span_costs_with_gold` rebuilds the tables and the sentences with their gold spans, which is everything phase B needs. |
| `ModuleNotFoundError: m1_analyzer` | Re-run cell 2, or *Runtime → Restart session* and run all again. |

Full details: `architecture.md`, `docs/design_decisions.md`, `docs/edge_cases.md`, and the theory
background `theory_1b_bracket_induction.md`.